In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import lightgbm as lgbm
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as mse
from sklearn.datasets import fetch_california_housing

## Overview

Here is a simple Light GBM Baseline:
* Combining Kaggle train with external data
* Training and evaluating an LGBM model on 10 folds
* Submitting the average test-predictions of the 10 model

Here is my [EDA](https://www.kaggle.com/code/soupmonster/s03e01-eda-for-modelling) with a focus on modelling. 

## Data Loading

In [ ]:
path = "/kaggle/input/playground-series-s3e1/"
train = pd.read_csv(path+"train.csv")
test = pd.read_csv(path+"test.csv")
sub = pd.read_csv(path+"sample_submission.csv")

In [ ]:
features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population','AveOccup', 'Latitude', 'Longitude']
target = 'MedHouseVal'

In [ ]:
#combine with original dataset
#ref: https://www.kaggle.com/code/phongnguyen1/s03e01-original-data-boost-automl?scriptVersionId=115366781

external_data = fetch_california_housing()
train2 = pd.DataFrame(external_data['data'])
train2[target] = external_data['target']
train2.columns = features+[target]
train_all = pd.concat([train,train2],axis=0).drop_duplicates().reset_index(drop=True)

In [ ]:
train_all['is_generated'] = 0
train_all.loc[train.index,'is_generated']= 1


In [ ]:
inter_feats =[]
for f in ['Longitude','Latitude']:
    for f2 in ['MedInc','HouseAge','AveOccup']:
        if f!=f2:
            train_all[f"{f}_{f2}"] = train_all[f]*train_all[f2]
            test[f"{f}_{f2}"] = test[f]*test[f2]
            inter_feats.append(f"{f}_{f2}")

In [ ]:
features += inter_feats+['is_generated']
test['is_generated']= 1

In [ ]:
train_all.head()

## GBDT Model and CV

In [ ]:
k =10
kfold = KFold(k,shuffle=True, random_state=42)
val_scores = []
test_preds= []

for i,(train_idxs,val_idxs) in enumerate(kfold.split(train_all)):

    X_train = train_all.iloc[train_idxs][features]
    y_train = train_all.iloc[train_idxs][target]
    X_val = train_all.iloc[val_idxs][features]
    y_val = train_all.iloc[val_idxs][target]
    
    params= {
     'metric':'rmse',
     'learning_rate':0.02,
     'lambda_l1': 1.945,
     'num_leaves': 87,
     'feature_fraction': 0.79,
     'bagging_fraction': 0.93,
     'bagging_freq': 4,
     'min_data_in_leaf': 103,
     'max_depth': 17,
     'num_iterations':10000
    }
    
    model = lgbm.LGBMRegressor(**params)    
    
    model.fit(X= X_train,
              y= y_train,
              eval_set = (X_val,y_val),
              early_stopping_rounds =85,
              verbose=100
             )
    preds = model.predict(X_val)
    rmse = mse(y_val, preds,squared=False)
    val_scores.append(rmse)
    print(f'=== Fold {i} RMSE {rmse} ====')
    
    preds = model.predict(test[features])
    test_preds.append(preds)
    
print(f'=== Average RMSE of {k} Folds: {np.mean(val_scores)} ====')

In [ ]:
cv_feats = []
for i in model.feature_importances_.argsort()[::-1]:
    print(features[i], model.feature_importances_[i]/model.feature_importances_.sum())
    cv_feats.append(features[i])

## Submission

In [ ]:
preds = np.mean(np.vstack(test_preds),axis=0)
sub['MedHouseVal'] = preds
sub.to_csv("submission.csv", index=False)

Done. Have fun! 😊